In [1]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [2]:
input_data = "input_files/semantic_matching_0099.csv"
output_data = "output_files/semantic_matching_0099.xlsx"

In [3]:
# ## Open AI API
# os.environ["OPENAI_API_KEY"] = "your open ai api"
from dotenv import load_dotenv

load_dotenv()

True

## common functions

In [4]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [5]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()

        return content

In [6]:
"""
    This function is different a bit among different tasks, here is for the semantic matching task
"""

def construct_annotation(df, results, task_name="semantic_matching"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input = []
    outputByLLM = []
    
    for res in results:
        segmentation_name = res.get("segmentation_name")
        folder_name = res.get("folder_name")

        prefix_path = os.path.join(f"segmentation_filing/{task_name}", folder_name)
        real_segmentation_names = os.listdir(prefix_path)
        potential_seg_names = []

        ## obtain the potential segmentation folders using partial match
        for seg_name in real_segmentation_names:
            normalized_segmentation_name = segmentation_name.lower()
            normalized_seg_name = seg_name.lower()
            if (normalized_segmentation_name in normalized_seg_name) or (normalized_seg_name in normalized_segmentation_name):
                potential_seg_names.append(seg_name)
        
        parse = False
        cal = None
        pre = None
        defn = None
        lab = None
        ins = None
        sch = None
        seg_path = None
        answers = [tag.get("tag") for tag in res.get("error_tags")]
        for segmentation_name_ in potential_seg_names:
            
            full_path = os.path.join(prefix_path, segmentation_name_)
    
            temp_cal = None
            temp_pre = None
            temp_defn = None
            temp_lab = None
            temp_ins = None
            temp_sch = None
            
            file_list = os.listdir(full_path)
            
            for file_name in file_list:
                if file_name.endswith("cal.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_cal = read_xml(final_path)
                elif file_name.endswith("pre.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_pre = read_xml(final_path)
                elif file_name.endswith("def.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_defn = read_xml(final_path)
                elif file_name.endswith("lab.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_lab = read_xml(final_path)
                elif file_name.endswith("htm.xml"):
                    final_path = os.path.join(full_path,file_name)
                    temp_ins = read_xml(final_path)
                elif file_name.endswith("xsd"):
                    final_path = os.path.join(full_path,file_name)
                    temp_sch = read_xml(final_path)
                else:
                    continue

            is_appear = True
            for ans in answers:
                # check the presentation linkbase
                if ans in temp_pre:
                    continue
                elif ans.replace(":","") in temp_pre:
                    continue
                elif ans.replace(":","_") in temp_pre:
                    continue
                else:
                    is_appear = False
                    break
           
            if is_appear:
                parse = True
                cal = temp_cal
                pre = temp_pre
                defn = temp_defn
                lab = temp_lab
                ins = temp_ins
                sch = temp_sch
                seg_path = os.path.join(folder_name, segmentation_name_)
                break
        
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input.append(None)
        outputByLLM.append(res)
     
        
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for semantic matching task

In [7]:
sm_99_df = pd.read_csv(input_data)

In [8]:
sm_99_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0099.9533,"2022-01-03 11:15:38,239 [DQC.US.0099.9533] \nT...",2022-01-03,ERROR,DQC,99,1570132,"ANVI GLOBAL HOLDINGS, INC.",2022-01-03 10:40:00,0001553350-22-000002,US GAAP 2021,Field: Set; Name: xdx; ID: xdx_050_edei%2D%2DD...,10-Q,http://www.sec.gov/Archives/edgar/data/1570132...,2022-01-03,2022,409852,https://www.sec.gov/Archives/edgar/data/157013...,5400,10q-anvi-20211130
1,DQC.US.0099.9533,"2022-01-18 07:01:54,632 [DQC.US.0099.9533] \nT...",2022-01-18,ERROR,DQC,99,1796949,"GenFlat Holdings, Inc.",2022-01-18 06:02:00,0001654954-22-000549,US GAAP 2020,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1796949...,2022-01-18,2022,412314,https://www.sec.gov/Archives/edgar/data/179694...,8742,10q-hbr-20211130
2,DQC.US.0099.9533,"2022-01-18 07:01:54,632 [DQC.US.0099.9533] \nT...",2022-01-18,ERROR,DQC,99,1796949,"GenFlat Holdings, Inc.",2022-01-18 06:02:00,0001654954-22-000549,US GAAP 2020,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1796949...,2022-01-18,2022,412314,https://www.sec.gov/Archives/edgar/data/179694...,8742,10q-hbr-20211130
3,DQC.US.0099.9533,"2022-03-09 20:16:45,117 [DQC.US.0099.9533] \nT...",2022-03-10,ERROR,DQC,99,1304161,"PHARMA-BIO SERV, INC.",2022-02-11 17:21:00,0001654954-22-001532,US GAAP 2021,Document Created by XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1304161...,2022-02-11,2022,418188,https://www.sec.gov/Archives/edgar/data/130416...,8742,10k-pbsv-20211031
4,DQC.US.0099.9533,"2022-02-15 07:30:54,506 [DQC.US.0099.9533] \nT...",2022-02-15,ERROR,DQC,99,1620749,PANAMERA HOLDINGS CORPORATION,2022-02-14 21:51:00,0001640334-22-000315,US GAAP 2021,Document Created by XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1620749...,2022-02-15,2022,419692,https://www.sec.gov/Archives/edgar/data/162074...,8742,10k-pnht-20210731
5,DQC.US.0099.9533,"2022-02-15 07:30:54,507 [DQC.US.0099.9533] \nT...",2022-02-15,ERROR,DQC,99,1620749,PANAMERA HOLDINGS CORPORATION,2022-02-14 21:51:00,0001640334-22-000315,US GAAP 2021,Document Created by XBRLMaster,10-K,http://www.sec.gov/Archives/edgar/data/1620749...,2022-02-15,2022,419692,https://www.sec.gov/Archives/edgar/data/162074...,8742,10k-pnht-20210731
6,DQC.US.0099.9533,"2022-02-18 09:45:49,004 [DQC.US.0099.9533] \nT...",2022-02-18,ERROR,DQC,99,1602813,FRONTERA GROUP INC.,2022-02-18 09:24:00,0001549727-22-000009,US GAAP 2020,Field: Doc-Info; Name: Generator; Value: GoFil...,10-Q,http://www.sec.gov/Archives/edgar/data/1602813...,2022-02-18,2022,422094,https://www.sec.gov/Archives/edgar/data/160281...,8742,10q-frtg-20211231
7,DQC.US.0099.9533,"2022-12-19 12:02:34,485 [DQC.US.0099.9533] \nT...",2022-12-19,ERROR,DQC,99,1230524,"Leopard Energy, Inc.",2022-12-14 16:20:00,0001903596-22-000915,US GAAP 2021,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/1230524...,2022-12-14,2022,535328,https://www.sec.gov/Archives/edgar/data/123052...,8742,10q-cetc-20221031
8,DQC.US.0099.9533,"2022-11-23 06:02:44,800 [DQC.US.0099.9533] \nT...",2022-11-23,ERROR,DQC,99,1602813,FRONTERA GROUP INC.,2022-11-22 15:10:00,0001549727-22-000096,US GAAP 2022,Novaworks,10-Q,http://www.sec.gov/Archives/edgar/data/1602813...,2022-11-22,2022,531482,https://www.sec.gov/Archives/edgar/data/160281...,8742,10q-frtg-20220930
9,DQC.US.0099.9533,"2022-11-17 06:28:43,249 [DQC.US.0099.9533] \nT...",2022-11-17,ERROR,DQC,99,1876945,Gold Flora Corporation,2022-11-14 16:56:00,0001654954-22-015300,US GAAP 2022,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1876945...,2022-11-14,2022,529878,https://www.sec.gov/Archives/edgar/data/187694...,100,10q-gramf-20220930


## LLM judge assisstant

In [9]:
Client = OpenAI()

In [11]:
sm_99_system_prompt = """You are given a DQC validation message. Your task is to extract two specific pieces of information:

1. **Statement Name**: Extract the name of the financial statement mentioned in the message. If it appears in a phrase like "0000007 - Statement - Consolidated Statements of Operations", ignore any prefixes such as "- Statement -" or ID numbers. Only return the clean name of the statement (e.g., "Consolidated Statements of Operations").

2. **Erroneous us-gaap Tag(s)**: From all `us-gaap:` prefixed tags mentioned in the message, extract only those that are clearly described as having an error or violation. A tag should be included **only if** it satisfies both of the following:
   - It is explicitly stated to be **not included in any calculation relationship**.
   - The message states that it **will produce an error**, **violates a rule**, or **should have been reported differently**.
Do not include `us-gaap:` tags that are merely suggested, referenced as valid abstract parents, or shown as examples for correction.

3. **Reasoning**: For each extracted erroneous tag, provide a short explanation summarizing why it was classified as erroneous, based on the message.

Output your result as a structured JSON in the following format:
```json
{
  "statement_name": "...",
  "error_tags": [
    {
      "tag": "...",
      "reason": "..."
    }
  ]
}
"""

In [12]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": sm_99_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [13]:
res = []
for _, row in tqdm(sm_99_df.iterrows()):
    ## load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    ## load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    ## covert the extracted statement name in the message to PascalCase format as the segmentation name
    segmentation_name = to_pascal_case(response.get("statement_name"))

    ## record the folder name
    response["folder_name"] = folder_name
    ## record the extracted segmenataion name
    response["segmentation_name"] = segmentation_name
    res.append(response)

0it [00:00, ?it/s]

51it [02:22,  2.79s/it]


In [14]:
res[0]

{'statement_name': 'CONDENSED BALANCE SHEETS',
 'error_tags': [{'tag': 'us-gaap:PreferredStockValue',
   'reason': 'The tag is included in the presentation but not in any calculation relationship, which will produce an error.'}],
 'folder_name': '10q-anvi-20211130',
 'segmentation_name': 'CondensedBalanceSheets'}

In [15]:
"""
    the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
"""
new_sm_99_df = construct_annotation(sm_99_df, res, task_name="semantic_matching")

In [21]:
new_sm_99_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0099.9533,"2022-01-03 11:15:38,239 [DQC.US.0099.9533] \nT...",2022-01-03,ERROR,DQC,99,1570132,"ANVI GLOBAL HOLDINGS, INC.",2022-01-03 10:40:00,0001553350-22-000002,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:PreferredStockValue],10q-anvi-20211130/CondensedBalanceSheets,True,"{'statement_name': 'CONDENSED BALANCE SHEETS',..."
1,DQC.US.0099.9533,"2022-01-18 07:01:54,632 [DQC.US.0099.9533] \nT...",2022-01-18,ERROR,DQC,99,1796949,"GenFlat Holdings, Inc.",2022-01-18 06:02:00,0001654954-22-000549,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[],10q-hbr-20211130/ConsolidatedStatementsOfOpera...,True,{'statement_name': 'Consolidated Statements of...
2,DQC.US.0099.9533,"2022-01-18 07:01:54,632 [DQC.US.0099.9533] \nT...",2022-01-18,ERROR,DQC,99,1796949,"GenFlat Holdings, Inc.",2022-01-18 06:02:00,0001654954-22-000549,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[us-gaap:ProceedsFromNotesPayable, us-gaap:Pro...",10q-hbr-20211130/ConsolidatedStatementsOfCashF...,True,{'statement_name': 'Consolidated Statements of...
3,DQC.US.0099.9533,"2022-03-09 20:16:45,117 [DQC.US.0099.9533] \nT...",2022-03-10,ERROR,DQC,99,1304161,"PHARMA-BIO SERV, INC.",2022-02-11 17:21:00,0001654954-22-001532,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:OtherComprehensiveIncomeForeignCurren...,10k-pbsv-20211031/ConsolidatedStatementsOfComp...,True,{'statement_name': 'Consolidated Statements of...
4,DQC.US.0099.9533,"2022-02-15 07:30:54,506 [DQC.US.0099.9533] \nT...",2022-02-15,ERROR,DQC,99,1620749,PANAMERA HOLDINGS CORPORATION,2022-02-14 21:51:00,0001640334-22-000315,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,"[us-gaap:LiabilitiesAndStockholdersEquity, us-...",10k-pnht-20210731/BalanceSheets,True,"{'statement_name': 'Balance Sheets', 'error_ta..."
5,DQC.US.0099.9533,"2022-02-15 07:30:54,507 [DQC.US.0099.9533] \nT...",2022-02-15,ERROR,DQC,99,1620749,PANAMERA HOLDINGS CORPORATION,2022-02-14 21:51:00,0001640334-22-000315,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[us-gaap:CashAndCashEquivalentsPeriodIncreaseD...,10k-pnht-20210731/StatementsOfCashFlowsUnaudited,True,{'statement_name': 'Statements of Cash Flows (...
6,DQC.US.0099.9533,"2022-02-18 09:45:49,004 [DQC.US.0099.9533] \nT...",2022-02-18,ERROR,DQC,99,1602813,FRONTERA GROUP INC.,2022-02-18 09:24:00,0001549727-22-000009,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,None,None,None,[us-gaap:IncreaseDecreaseInAccountsPayableAndA...,10q-frtg-20211231/StatementsOfCashFlows,True,{'statement_name': 'STATEMENTS OF CASH FLOWS (...
7,DQC.US.0099.9533,"2022-12-19 12:02:34,485 [DQC.US.0099.9533] \nT...",2022-12-19,ERR

In [20]:
new_sm_99_df.to_excel(output_data,index=False)